# Clean Olist Dataset

This notebook uses the `dd_cleaner.notebook_utils` APIs to load the featurization input dataset from the KMDS workspace and process it for the SP 2017 clustering workflow.


## Input source and outputs

- The featurization input dataset is resolved through `dd_cleaner.notebook_utils` and the workspace `config.yaml`.
- Outputs are written into the flat `data/` directory with `_prepared` suffixes.

Outputs produced by this notebook:
- `data/SP_2017_orders_filtered_prepared.csv`
- `data/SP_2017_weekly_revenue_prepared.csv`
- `data/SP_2017_freq_prod_weekly_sales_prepared.csv`
- `data/SP_2017_freq_prod_weekly_sales_prepared.parquet`


In [6]:
import pandas as pd
from pathlib import Path

from dd_cleaner.notebook_utils import (
    init_notebook_session,
    get_raw_data,
    get_cleaned_data,
    get_tagged_entities,
    get_attributes_by_tag,
)

coord, artifacts_df = init_notebook_session("..")
artifacts_df


✅ Notebook session initialized for workspace: /home/rajiv/programming/kmds_migration/olist_migration

Available Artifacts:

Artifact Name  \
0                         Raw Data   
1                     Cleaned Data   
2                User Cleaned Data   
3             Tagged Entities (DD)   
4  Cleaning Recommendations Report   
5                 Profiling Report   
6                   Handshake File   
7                  Quarantine File   
8               Metadata Authority   

                                          File Name  \
0                   olist_daily_orders_prepared.csv   
1             olist_daily_orders_prepared_clean.csv   
2      olist_daily_orders_prepared_user_cleaned.csv   
3  olist_daily_orders_prepared_analysis_results.csv   
4                       cleaning_recommendations.md   
5   olist_daily_orders_prepared_profiling_report.md   
6                       parser_cleaner_handshake.md   
7        olist_daily_orders_prepared_quarantine.csv   
8    olist_daily_orders_prepared_metadata_table.csv   

                                            Location  Exists  
0               data/olist_daily_orders_prepared.csv    True  
1  data/dd_cleaner/olist_daily_orders_prepared_cl...    True  
2  data/dd_cleaner/olist_daily_orders_prepared_us...   False  
3  documents/dd_analysis_results/olist_daily_orde...    True  
4   documents/dd_cleaner/cleaning_recommendations.md    True  
5  documents/dd_cleaner/olist_daily_orders_prepar...    True  
6   documents/dd_cleaner/parser_cleaner_handshake.md    True  
7  data/quarantine/olist_daily_orders_prepared_qu...   False  
8  data/dd_cleaner/olist_daily_orders_prepared_me...   False

,Artifact Name,File Name,Location,Exists
0,Raw Data,olist_daily_orders_prepared.csv,data/olist_daily_orders_prepared.csv,True
1,Cleaned Data,olist_daily_orders_prepared_clean.csv,data/dd_cleaner/olist_daily_orders_prepared_cl...,True
2,User Cleaned Data,olist_daily_orders_prepared_user_cleaned.csv,data/dd_cleaner/olist_daily_orders_prepared_us...,False
3,Tagged Entities (DD),olist_daily_orders_prepared_analysis_results.csv,documents/dd_analysis_results/olist_daily_orde...,True
4,Cleaning Recommendations Report,cleaning_recommendations.md,documents/dd_cleaner/cleaning_recommendations.md,True
5,Profiling Report,olist_daily_orders_prepared_profiling_report.md,documents/dd_cleaner/olist_daily_orders_prepar...,True
6,Handshake File,parser_cleaner_handshake.md,documents/dd_cleaner/parser_cleaner_handshake.md,True
7,Quarantine File,olist_daily_orders_prepared_quarantine.csv,data/quarantine/olist_daily_orders_prepared_qu...,False
8,Metadata Authority,olist_daily_orders_prepared_metadata_table.csv,data/dd_cleaner/olist_daily_orders_prepared_me...,False


In [7]:
df_raw = get_raw_data(coord)
df_cleaned = get_cleaned_data(coord)
df_tags = get_tagged_entities(coord)
geographic_attributes = get_attributes_by_tag(coord, "geographic")

print("Raw dataset shape:", df_raw.shape)
print("Cleaned dataset shape:", df_cleaned.shape)
print("Tagged entities shape:", df_tags.shape)
print("Geographic attributes:", geographic_attributes)
df_raw.head(3)


Raw dataset shape: (112650, 11)
Cleaned dataset shape: (112650, 11)
Tagged entities shape: (11, 7)
Geographic attributes: ['customer_zip_code_prefix', 'customer_city', 'customer_state']


,order_id,customer_id,order_purchase_timestamp,order_item_id,product_id,price,customer_zip_code_prefix,customer_city,customer_state,freq_cust,freq_purch_prod
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,2017-09-13 08:59:02,1,4244733e06e7ecb4970a6e2683c13e61,58.9,28013,campos dos goytacazes,RJ,False,True
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,2017-04-26 10:53:06,1,e5f2d52b802189ee658865ca93d83a8f,239.9,15775,santa fe do sul,SP,False,False
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,2018-01-14 14:33:31,1,c777355d18b72b67abbeef9df44fd0fd,199.0,35661,para de minas,MG,False,False


## Step 1: Inspect the featurization input

The raw dataset loaded by `get_raw_data()` is the featurization input anchor for this workspace. We inspect its schema and ensure the timestamp and geographic fields are present.


In [8]:
print("Columns:", df_raw.columns.tolist())
print(df_raw.dtypes)
df_raw.sample(5, random_state=42)


Columns: ['order_id', 'customer_id', 'order_purchase_timestamp', 'order_item_id', 'product_id', 'price', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'freq_cust', 'freq_purch_prod']
order_id                        str
customer_id                     str
order_purchase_timestamp        str
order_item_id                 int64
product_id                      str
price                       float64
customer_zip_code_prefix      int64
customer_city                   str
customer_state                  str
freq_cust                      bool
freq_purch_prod                bool
dtype: object


,order_id,customer_id,order_purchase_timestamp,order_item_id,product_id,price,customer_zip_code_prefix,customer_city,customer_state,freq_cust,freq_purch_prod
107777,f4ee4273538924bda6212f5948e80fde,8871d0db2013a98a58576b50ff9a1c1d,2018-06-12 08:59:54,1,69455f41626a745aea9ee9164cb9eafd,180.00,28943,sao pedro da aldeia,RJ,False,True
2391,056349f85a73d794119c4286c95a52de,03bbe0ce5c28e05f22917607db798818,2017-03-03 14:21:58,1,af35be35db4ad0dc288b571453337376,10.99,72465,brasilia,DF,False,True
77829,b124967afcc82ef17ec41020fe2a9136,43dd5cdd60809c5f268d9d3b1e5b986d,2018-05-29 16:19:32,1,12e6d0f655986ceff00c74658dec97b1,49.99,5311,sao paulo,SP,False,False
99819,e257ae8610fb4fb68a1f459c3a4b1f51,b46f11e377a570f09af36515191b1c1e,2017-05-13 07:49:29,1,a50acd33ba7a8da8e9db65094fa990a4,117.30,9170,santo andre,SP,False,True
41297,5e114d8e3840661abc3d9c4820f427b3,b7bf7024fbd5f37207415a82f73d70e6,2018-04-26 18:01:20,1,5cca3efb9521cc1d7099d610d4a12017,58.90,9715,sao bernardo do campo,SP,False,False


## Step 2: Derive SP 2017 subset

Use the loaded featurization input dataset and derive the SP 2017 subset required by the clustering notebook.


In [9]:
df_raw["order_purchase_timestamp"] = pd.to_datetime(df_raw["order_purchase_timestamp"])
df_raw["year"] = df_raw["order_purchase_timestamp"].dt.year
df_raw["month"] = df_raw["order_purchase_timestamp"].dt.month
df_raw["woy"] = df_raw["order_purchase_timestamp"].dt.isocalendar().week

df_sp_2017 = df_raw[(df_raw["customer_state"] == "SP") & (df_raw["year"] == 2017)].reset_index(drop=True)
output_dir = Path("../data")
sp_orders_path = output_dir / "SP_2017_orders_filtered_prepared.csv"
df_sp_2017.to_csv(sp_orders_path, index=False)
print("SP 2017 subset saved to:", sp_orders_path)
print(df_sp_2017.shape)
df_sp_2017.head(5)


SP 2017 subset saved to: ../data/SP_2017_orders_filtered_prepared.csv
(20044, 14)


,order_id,customer_id,order_purchase_timestamp,order_item_id,product_id,price,customer_zip_code_prefix,customer_city,customer_state,freq_cust,freq_purch_prod,year,month,woy
0,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,2017-04-26 10:53:06,1,e5f2d52b802189ee658865ca93d83a8f,239.90,15775,santa fe do sul,SP,False,False,2017,4,17
1,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,2017-02-04 13:57:51,1,ac6c3623068f30de03045865e4e10089,199.90,13226,varzea paulista,SP,False,True,2017,2,5
2,00054e8431b9d7675808bcb819fb4a32,32e2e6ab09e778d99bf2e0ecd4898718,2017-12-10 11:53:48,1,8d4f2bb7e93e6710a28f34fa83ee7d28,19.90,16700,guararapes,SP,False,False,2017,12,49
3,000c3e6612759851cc3cbb4b83257986,3773bcf1a6fbd29233ea1c1b573c4f22,2017-08-12 10:08:57,1,b50c950aba0dcead2c48032a690ce817,99.00,13208,jundiai,SP,False,False,2017,8,32
4,000e906b789b55f64edcb1f84030f90d,6a3b2fc9f270df258605e22bef19fd88,2017-11-21 18:54:23,1,57d79905de06d8897872c551bfd09358,21.99,18900,santa cruz do rio pardo,SP,False,True,2017,11,47


## Step 3: Create SP 2017 weekly revenue and product-week matrix

Aggregate the SP 2017 subset into the revenue and product-week artifacts used by the final clustering notebook.


In [10]:
# 5) Save weekly SP product matrix outputs
sp_freq_prod_path = output_dir / "SP_2017_freq_prod_weekly_sales_prepared.csv"
sp_freq_prod_parquet = output_dir / "SP_2017_freq_prod_weekly_sales_prepared.parquet"

print("Saving product-week matrix CSV to:", sp_freq_prod_path)
df_freq_prod.to_csv(sp_freq_prod_path, index=False)

try:
    df_freq_prod.to_parquet(sp_freq_prod_parquet, index=False)
    print("Product-week parquet saved to:", sp_freq_prod_parquet)
except ImportError as err:
    print("Warning: parquet support is unavailable:", err)
    print("CSV output was written successfully. Install pyarrow or fastparquet to enable parquet export.")

print("Product-week matrix saved to:", sp_freq_prod_path)
df_freq_prod.head(5)

Saving product-week matrix CSV to: ../data/SP_2017_freq_prod_weekly_sales_prepared.csv
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.
CSV output was written successfully. Install pyarrow or fastparquet to enable parquet export.
Product-week matrix saved to: ../data/SP_2017_freq_prod_weekly_sales_prepared.csv


product_id,woy,00088930e925c41fd95ebfe695fd2655,0009406fd7479715e4bef61dd91f2462,00126f27c813603687e6ce486d909d01,001795ec6f1b187d37335e1c4704762e,001b72dfd63e9833e8c02742adf472e3,00210e41887c2a8ef9f791ebc780cc36,0021a87d4997a48b6cef1665602be0f5,00250175f79f584c14ab5cecd80553cd,0030026a6ddb3b2d1d4bc225b4b4c4da,...,ffbceea72c6f921df4bb547275b9ca14,ffbe3df3856b1fef3fee8f1264105a89,ffc0b406806006602c5853b00ab5f7fd,ffc48c754b5bd736e2887e279d1dec72,ffc9d90bae2127e6a6ce6d6654267ebd,ffce5ed9e0bcc2e46796b988cdac733b,ffd4bf4306745865e5692f69bd237893,ffe8083298f95571b4a66bfbc1c05524,fff0a542c3c62682f23305214eaeaa24,fff9553ac224cec9d15d49f5a263411f
0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Validation

Confirm the generated files exist and inspect their shapes.


In [11]:
paths = [
    sp_orders_path,
    sp_weekly_revenue_path,
    sp_freq_prod_path,
    sp_freq_prod_parquet,
]
for path in paths:
    print(path, "exists=", path.exists())
    if path.exists() and path.suffix == ".csv":
        df = pd.read_csv(path)
        print(f"  {path.name}: {len(df)} rows, {len(df.columns)} cols")


../data/SP_2017_orders_filtered_prepared.csv exists= True
  SP_2017_orders_filtered_prepared.csv: 20044 rows, 14 cols
../data/SP_2017_weekly_revenue_prepared.csv exists= True
  SP_2017_weekly_revenue_prepared.csv: 52 rows, 3 cols
../data/SP_2017_freq_prod_weekly_sales_prepared.csv exists= True
  SP_2017_freq_prod_weekly_sales_prepared.csv: 52 rows, 9368 cols
../data/SP_2017_freq_prod_weekly_sales_prepared.parquet exists= False
